# **Session 2 : LangChain**

####LLM API Call without Framework

In [ ]:
%pip install openai groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.4/131.4 kB 4.6 MB/s eta 0:00:00


In [ ]:
import os
import getpass
from openai import OpenAI

Using OpenAI

In [ ]:
os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API Key:")

KeyboardInterrupt: Interrupted by user

In [ ]:
openai_direct_model = OpenAI()

In [ ]:
openai_direct_response = openai_direct_model.chat.completions.create(
    model="gpt-3.5-turbo",
    temperature=1,
    messages=[
        {"role": "system", "content": "You are an Egyptian TV football news analyst assistant. You answer the user’s questions with at least two memorable match moments."},
        {"role": "user", "content": "how many Egyptian football won African Cup?"}
        ]
    )
openai_direct_response

In [ ]:
print(openai_direct_response.choices[0].message.content)

Using Groq Models

In [ ]:
from groq import Groq

In [ ]:
os.environ["GROQ_API_KEY"] = getpass.getpass("Groq API Key:")

In [ ]:
groq_direct_model = Groq()

In [ ]:
groq_direct_response = groq_direct_model.chat.completions.create(
    model="moonshotai/kimi-k2-instruct",
    temperature=1,
    messages=[
        {"role": "system", "content": "You are an Egyptian TV football news analyst assistant. You answer the user’s questions with at least two memorable match moments."},
        {"role": "user", "content": "how many Egyptian football won African Cup?"}
        ]
    )
groq_direct_response

In [ ]:
print(groq_direct_response.choices[0].message.content)

# LangChain Framework
Install langchain and langchain integration for OpenAI and Groq

In [ ]:
%pip install langchain langchain-openai langchain-Groq

# **1) Lets try first simple direct call ChatModel**
### OpenAI

In [ ]:
import os
import getpass
from langchain_openai import ChatOpenAI

In [ ]:
os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API Key:")

In [ ]:
openai_model = ChatOpenAI() # defulat is openai gpt3.5 turbo

In [ ]:
respond = openai_model.invoke("what is the captial of Egypt?")
respond

In [ ]:
print(respond.content)

### Groq and let's pick a model from Groq

In [ ]:
from langchain_groq import ChatGroq

In [ ]:
os.environ["GROQ_API_KEY"] = getpass.getpass("Groq API Key:")

In [ ]:
groq_model = ChatGroq(model="moonshotai/kimi-k2-instruct") #you must pick a model

In [ ]:
respond = groq_model.invoke("what is the captial of Egypt?")
respond

In [ ]:
print(respond.content)

# **2)Messages, Parser**

## Messages

In [ ]:
from langchain_core.messages import HumanMessage, SystemMessage

In [ ]:
Input_Masseage = [
    SystemMessage(content="You are an Egyptian TV football news analyst assistant. You answer the user’s questions with at least two memorable match moments."),
    HumanMessage(content="how many Egyptian football won African Cup?")
    ]
Input_Masseage

In [ ]:
openai_respond = openai_model.invoke(Input_Masseage)
openai_respond

In [ ]:
print(openai_respond.content)

In [ ]:
groq_respond = groq_model.invoke(Input_Masseage)
groq_respond

In [ ]:
print(groq_respond.content)

##*String* Output - Parser

In [ ]:
from langchain_core.output_parsers import StrOutputParser

In [ ]:
Parser = StrOutputParser()

In [ ]:
pares_openai_respond = Parser.invoke(openai_respond)
print(pares_openai_respond)

In [ ]:
pares_groq_respond = Parser.invoke(groq_respond)
print(pares_groq_respond)

## Prompt Tempalte

In [ ]:
from langchain_core.prompts import ChatPromptTemplate, SystemMessagePromptTemplate, HumanMessagePromptTemplate

In [ ]:
#Desing Template with variables

prompt_template = ChatPromptTemplate.from_messages(
    [
        SystemMessagePromptTemplate.from_template("You are an Egyptian TV football news analyst assistant. You answer the user’s questions with at least two memorable match moments."),
        HumanMessagePromptTemplate.from_template("how many {team_nationality} football team won {continental} Cup?")
        ]
    )
prompt_template

In [ ]:
prompt_template_with_Var = prompt_template.invoke({"team_nationality": "Egyptian", "continental": "African"})
prompt_template_with_Var

In [ ]:
prompt_template_respond = groq_model.invoke(prompt_template_with_Var)
prompt_template_respond

In [ ]:
prompt_template_resp_parse = Parser.invoke(prompt_template_respond)
print(prompt_template_resp_parse)

###Chain

In [ ]:
#First Chain
chain1 = prompt_template | groq_model | Parser
chain1

In [ ]:
chain1_respond = chain1.invoke({"team_nationality": "Egyptian", "continental": "African"})
print(chain1_respond)

In [ ]:
prompt_template2 = ChatPromptTemplate.from_messages(
    [
        SystemMessagePromptTemplate.from_template("You are an Egyptian TV translator assistant. Your job is to translate english from footballer analyzer to Egyptian Arabic Audience."),
        HumanMessagePromptTemplate.from_template("Analyzed topic: {chain_1_output}")
        ]
    )
prompt_template2

In [ ]:
model_chain2 = ChatGroq(model="openai/gpt-oss-120b")

In [ ]:
Parser2 = StrOutputParser()

In [ ]:
chain2 = prompt_template2 | model_chain2 | Parser2
chain2

In [ ]:
chain2_response = chain2.invoke({"chain_1_output": chain1_respond})
print(chain2_response)

In [ ]:
prompt_template3 = ChatPromptTemplate.from_messages(
    [
        SystemMessagePromptTemplate.from_template("You are an Egyptian A journalist who writes articles for a sports newspaper facebook page."),
        HumanMessagePromptTemplate.from_template("geneate a sport post about {chain_2_output}")
        ]
    )
prompt_template3

In [ ]:
model_chain3 = ChatGroq(model="openai/gpt-oss-120b")

In [ ]:
Parser3 = StrOutputParser()

In [ ]:
chain3 = prompt_template3 | model_chain3 | Parser3
chain3

In [ ]:
chain3_response = chain3.invoke({"chain_2_output": chain2_response})
print(chain3_response)